In [3]:
"""
Dueling CNN Q-Network for Surface Code Decoding
=================================================

Input  : state tensor  shape (2k, 2d+1, 2d+1)
                             (6,  7,    7   )   for d=3, k=3

Output : Q-values      shape (n_actions,)
                             (19,)              for d=3

Architecture
------------

(6, 7, 7)
    │
    ▼
┌─────────────────────────────────┐
│  Conv2d(6  → 32, k=3, pad=1)   │  ← detect local syndrome patterns
│  BatchNorm2d(32)                │
│  ReLU                           │
├─────────────────────────────────┤
│  Conv2d(32 → 64, k=3, pad=1)   │  ← combine neighbouring patterns
│  BatchNorm2d(64)                │
│  ReLU                           │
├─────────────────────────────────┤
│  Conv2d(64 → 64, k=3, pad=1)   │  ← deep abstract features
│  BatchNorm2d(64)                │
│  ReLU                           │
└─────────────────────────────────┘
    │
    ▼
Flatten  →  (64 × 7 × 7 = 3136)
    │
    ├──────────────────────────────────────────────┐
    ▼                                              ▼
┌──────────────────────┐              ┌──────────────────────────┐
│   Value  stream      │              │   Advantage  stream      │
│   FC(3136 → 256)     │              │   FC(3136 → 256)         │
│   ReLU               │              │   ReLU                   │
│   FC(256  → 1)       │              │   FC(256  → n_actions)   │
│                      │              │                          │
│   V(s)  scalar       │              │   A(s,a)  vector         │
└──────────────────────┘              └──────────────────────────┘
    │                                              │
    └──────────────────┬───────────────────────────┘
                       ▼
         Q(s,a) = V(s) + A(s,a) - mean_a[ A(s,a) ]

Why dueling?
------------
Many steps the syndrome weight does NOT change regardless of which
correction is applied.  The Value stream learns "how good is this
syndrome state" independently of the action.  The Advantage stream
learns "which action is relatively better".  This separation makes
learning faster and more stable for our environment.

Why BatchNorm?
--------------
Syndrome grids are sparse (mostly zeros, few ±1 values).
BatchNorm prevents the activations from collapsing during early
training when the agent is exploring randomly.
"""

import torch
import torch.nn as nn
import numpy as np


class DuelingCNN(nn.Module):
    """
    Dueling CNN Q-Network.

    Parameters
    ----------
    in_channels : int   number of input channels  (= 2k = 6 for d=3)
    grid_size   : int   spatial size of the grid  (= 2d+1 = 7 for d=3)
    n_actions   : int   number of discrete actions (= 2d²+1 = 19 for d=3)
    """

    def __init__(self, in_channels: int, grid_size: int, n_actions: int):
        super().__init__()

        self.in_channels = in_channels
        self.grid_size   = grid_size
        self.n_actions   = n_actions

        # ── Convolutional backbone ────────────────────────────────────────────
        # padding=1 keeps spatial size unchanged (7→7→7)
        # so the flattened size is always 64 × grid_size × grid_size
        self.conv = nn.Sequential(

            # Layer 1 — local pattern detection
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # Layer 2 — combine neighbouring patterns
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # Layer 3 — deep abstract features
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

        # Flattened feature size: 64 channels × grid_size × grid_size
        flat_size = 64 * grid_size * grid_size   # = 3136 for d=3

        # ── Value stream  V(s) → scalar ──────────────────────────────────────
        self.value_stream = nn.Sequential(
            nn.Linear(flat_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1),               # single scalar
        )

        # ── Advantage stream  A(s,a) → vector of size n_actions ──────────────
        self.advantage_stream = nn.Sequential(
            nn.Linear(flat_size, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions),       # one value per action
        )

        # ── Weight initialisation ─────────────────────────────────────────────
        self._init_weights()

    # ── Forward pass ──────────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : torch.Tensor  shape (batch, in_channels, grid_size, grid_size)
                                (B,     6,           7,         7        )

        Returns
        -------
        q : torch.Tensor  shape (batch, n_actions)
                                (B,     19         )
        """
        # 1. Conv backbone
        features = self.conv(x)                          # (B, 64, 7, 7)

        # 2. Flatten
        features = features.view(features.size(0), -1)  # (B, 3136)

        # 3. Two streams
        V = self.value_stream(features)                  # (B, 1)
        A = self.advantage_stream(features)              # (B, 19)

        # 4. Dueling combination
        #    Q(s,a) = V(s) + A(s,a) - mean_a[ A(s,a) ]
        #    subtracting the mean makes A identifiable (zero-mean advantage)
        Q = V + A - A.mean(dim=1, keepdim=True)         # (B, 19)

        return Q

    # ── Convenience methods ───────────────────────────────────────────────────

    def get_q_values(self, state: torch.Tensor) -> torch.Tensor:
        """
        Get Q-values for a single state (no batch dimension needed).

        Parameters
        ----------
        state : torch.Tensor  shape (in_channels, grid_size, grid_size)

        Returns
        -------
        q : torch.Tensor  shape (n_actions,)
        """
        with torch.no_grad():
            return self.forward(state.unsqueeze(0)).squeeze(0)

    def get_action(self, state: torch.Tensor) -> int:
        """
        Greedy action selection for a single state.

        Parameters
        ----------
        state : torch.Tensor  shape (in_channels, grid_size, grid_size)

        Returns
        -------
        action : int
        """
        q = self.get_q_values(state)
        return int(q.argmax().item())

    # ── Weight initialisation ─────────────────────────────────────────────────

    def _init_weights(self):
        """
        He initialisation for ReLU layers.
        Zero-initialise the final advantage layer bias so Q-values
        start near zero rather than random large values.
        """
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                nn.init.zeros_(module.bias)

    # ── Summary ───────────────────────────────────────────────────────────────

    def summary(self):
        """Print a readable summary of shapes and parameter counts."""
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)

        flat_size = 64 * self.grid_size * self.grid_size

        print(f"\n{'═'*52}")
        print(f"  Dueling CNN Q-Network")
        print(f"{'═'*52}")
        print(f"  Input  : ({self.in_channels}, {self.grid_size}, {self.grid_size})")
        print(f"{'─'*52}")
        print(f"  Conv1  : ({self.in_channels}→32,  k=3, pad=1)  → (32, {self.grid_size}, {self.grid_size})")
        print(f"  Conv2  : (32→64, k=3, pad=1)  → (64, {self.grid_size}, {self.grid_size})")
        print(f"  Conv3  : (64→64, k=3, pad=1)  → (64, {self.grid_size}, {self.grid_size})")
        print(f"  Flatten: {flat_size}")
        print(f"{'─'*52}")
        print(f"  Value     : {flat_size}→256→1")
        print(f"  Advantage : {flat_size}→256→{self.n_actions}")
        print(f"{'─'*52}")
        print(f"  Output : Q(s,a)  shape ({self.n_actions},)")
        print(f"{'─'*52}")
        print(f"  Total params    : {total:,}")
        print(f"  Trainable params: {trainable:,}")
        print(f"{'═'*52}\n")


In [5]:
"""
Test Suite — Dueling CNN Q-Network
====================================
Verifies shapes, values, and gradients.
"""

PASS = "✅"
FAIL = "❌"

def check(name, cond, detail=""):
    status = PASS if cond else FAIL
    print(f"  {status} {name}" + (f"  [{detail}]" if detail else ""))
    return cond

# ── Parameters matching our d=3 environment ───────────────────────────────────
IN_CHANNELS = 6    # 2k = 2×3
GRID_SIZE   = 7    # 2d+1 = 2×3+1
N_ACTIONS   = 19   # 2×d²+1 = 2×9+1
BATCH       = 32

net = DuelingCNN(IN_CHANNELS, GRID_SIZE, N_ACTIONS)
net.summary()

# ──────────────────────────────────────────────────────────────────────────────
print("══════════════════════════════════════")
print(" TEST 1 — Output shape")
print("══════════════════════════════════════")

x_batch = torch.zeros(BATCH, IN_CHANNELS, GRID_SIZE, GRID_SIZE)
q_batch = net(x_batch)

check("Input  shape (32, 6, 7, 7)",   x_batch.shape == (BATCH, IN_CHANNELS, GRID_SIZE, GRID_SIZE),
      str(tuple(x_batch.shape)))
check("Output shape (32, 19)",         q_batch.shape == (BATCH, N_ACTIONS),
      str(tuple(q_batch.shape)))

# ──────────────────────────────────────────────────────────────────────────────
print("\n══════════════════════════════════════")
print(" TEST 2 — Single state helpers")
print("══════════════════════════════════════")

state = torch.zeros(IN_CHANNELS, GRID_SIZE, GRID_SIZE)

q_single = net.get_q_values(state)
check("get_q_values output shape (19,)",  q_single.shape == (N_ACTIONS,),
      str(tuple(q_single.shape)))

action = net.get_action(state)
check("get_action returns int",           isinstance(action, int))
check("get_action in valid range [0,18]", 0 <= action < N_ACTIONS,
      f"got {action}")

# ──────────────────────────────────────────────────────────────────────────────
print("\n══════════════════════════════════════")
print(" TEST 3 — Dueling decomposition")
print("══════════════════════════════════════")
# Q = V + A - mean(A)  →  mean_a[Q] == V
# So mean of Q-values across actions should equal V

x = torch.randn(1, IN_CHANNELS, GRID_SIZE, GRID_SIZE)

# Extract V and A manually
features = net.conv(x).view(1, -1)
V = net.value_stream(features)          # (1, 1)
A = net.advantage_stream(features)      # (1, 19)
Q = net(x)                              # (1, 19)

# Check: Q = V + A - mean(A)
Q_manual = V + A - A.mean(dim=1, keepdim=True)
check("Q = V + A - mean(A)  [manual check]",
      torch.allclose(Q, Q_manual, atol=1e-5))

# Check: mean_a[Q] ≈ V
mean_Q = Q.mean(dim=1, keepdim=True)
check("mean_a[Q] ≈ V(s)",
      torch.allclose(mean_Q, V, atol=1e-5),
      f"mean_Q={mean_Q.item():.4f}  V={V.item():.4f}")

# ──────────────────────────────────────────────────────────────────────────────
print("\n══════════════════════════════════════")
print(" TEST 4 — Gradient flow")
print("══════════════════════════════════════")
# Q-values must be differentiable — backprop must reach conv weights

optimizer = torch.optim.Adam(net.parameters(), lr=1e-4)
x  = torch.randn(BATCH, IN_CHANNELS, GRID_SIZE, GRID_SIZE)
q  = net(x)

# Dummy loss: MSE against zeros
loss = (q ** 2).mean()
optimizer.zero_grad()
loss.backward()

# Check that every parameter has a gradient
no_grad = [n for n, p in net.named_parameters() if p.grad is None]
check("All parameters have gradients",  len(no_grad) == 0,
      f"missing: {no_grad}" if no_grad else "all good")

conv1_grad = net.conv[0].weight.grad
check("Conv1 weight gradient is non-zero",
      conv1_grad is not None and conv1_grad.abs().sum().item() > 0)

# ──────────────────────────────────────────────────────────────────────────────
print("\n══════════════════════════════════════")
print(" TEST 5 — Different states → different Q-values")
print("══════════════════════════════════════")
# The network must be sensitive to input differences

state_clean  = torch.zeros(1, IN_CHANNELS, GRID_SIZE, GRID_SIZE)
state_noisy  = torch.randn(1, IN_CHANNELS, GRID_SIZE, GRID_SIZE)

q_clean  = net(state_clean)
q_noisy  = net(state_noisy)

check("Different inputs → different Q-values",
      not torch.allclose(q_clean, q_noisy))

# ──────────────────────────────────────────────────────────────────────────────
print("\n══════════════════════════════════════")
print(" TEST 6 — Real environment state")
print("══════════════════════════════════════")
# Feed an actual state from the environment through the network


from src.environments.surface3_env import SurfaceCodeEnv

env = SurfaceCodeEnv(distance=3, noise=0.01)
obs, _ = env.reset(seed=42)

state_t = torch.FloatTensor(obs)                        # (6, 7, 7)
q_vals  = net.get_q_values(state_t)                     # (19,)
best_action = net.get_action(state_t)

check("Real env state passes through network",  q_vals.shape == (N_ACTIONS,))
check("Best action is valid",                   0 <= best_action < N_ACTIONS,
      f"action={best_action}")
check("Q-values are finite",                    torch.isfinite(q_vals).all().item())

print(f"\n  Q-values: min={q_vals.min():.4f}  max={q_vals.max():.4f}  "
      f"mean={q_vals.mean():.4f}")
print(f"  Best action: {best_action}")

# ──────────────────────────────────────────────────────────────────────────────
print("\n══════════════════════════════════════")
print(" TEST 7 — eval vs train mode")
print("══════════════════════════════════════")
# BatchNorm behaves differently in train vs eval mode
# eval() should give deterministic output

net.eval()
with torch.no_grad():
    q1 = net(state_t.unsqueeze(0))
    q2 = net(state_t.unsqueeze(0))
check("eval mode is deterministic",  torch.allclose(q1, q2))

net.train()
check("back to train mode",  net.training)

print("\n══════════════════════════════════════")
print(" ALL TESTS COMPLETE")
print("══════════════════════════════════════\n")


════════════════════════════════════════════════════
  Dueling CNN Q-Network
════════════════════════════════════════════════════
  Input  : (6, 7, 7)
────────────────────────────────────────────────────
  Conv1  : (6→32,  k=3, pad=1)  → (32, 7, 7)
  Conv2  : (32→64, k=3, pad=1)  → (64, 7, 7)
  Conv3  : (64→64, k=3, pad=1)  → (64, 7, 7)
  Flatten: 3136
────────────────────────────────────────────────────
  Value     : 3136→256→1
  Advantage : 3136→256→19
────────────────────────────────────────────────────
  Output : Q(s,a)  shape (19,)
────────────────────────────────────────────────────
  Total params    : 1,668,788
  Trainable params: 1,668,788
════════════════════════════════════════════════════

══════════════════════════════════════
 TEST 1 — Output shape
══════════════════════════════════════
  ✅ Input  shape (32, 6, 7, 7)  [(32, 6, 7, 7)]
  ✅ Output shape (32, 19)  [(32, 19)]

══════════════════════════════════════
 TEST 2 — Single state helpers
═══════════════════════════════

ModuleNotFoundError: No module named 'src'